# Notebook 00 — Merge Two SWH NetCDF Files

Combines two 18-month CMEMS SWH files into one 36-month file.

**Input:** Two `.nc` files (Jan 2021–Jun 2022) + (Jul 2022–Dec 2023)  
**Output:** `swh_daily_indian_ocean_2021_2023.nc` (single merged file)

**Run this first**, before any model notebook.

Set `RUN_ENV='colab'` if on Google Colab.

In [ ]:
# Cell 1 — Environment
RUN_ENV  = 'colab'
BASE_DIR = r'C:/BMP_Data/'
if RUN_ENV == 'colab':
    from google.colab import drive; drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BMP_Data/'

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

print("xarray version:", xr.__version__)

In [ ]:
# Cell 2 — Set file paths
# Both files should be in the same folder.
# FILE_1 = Jan 2021 – Jun 2022 (first 18 months)
# FILE_2 = Jul 2022 – Dec 2023 (second 18 months)
# Update these paths to match your actual filenames.

FILE_1   = os.path.join(BASE_DIR, 'swh_part1_2021_2022.nc')   # <── your first file
FILE_2   = os.path.join(BASE_DIR, 'swh_part2_2022_2023.nc')   # <── your second file
OUT_FILE = os.path.join(BASE_DIR, 'swh_daily_indian_ocean_2021_2023.nc')

# ── Quick sanity check ────────────────────────────────────────
for f in [FILE_1, FILE_2]:
    if not os.path.exists(f):
        print(f"NOT FOUND: {f}")
    else:
        size_mb = os.path.getsize(f) / 1024**2
        print(f"Found: {os.path.basename(f)}  ({size_mb:.1f} MB)")

In [ ]:
# Cell 3 — Inspect both files before merging
for label, fpath in [('FILE 1', FILE_1), ('FILE 2', FILE_2)]:
    print(f"\n{'='*55}")
    print(f"  {label}: {os.path.basename(fpath)}")
    print(f"{'='*55}")
    ds = xr.open_dataset(fpath)
    print(ds)
    print("\nData variables:", list(ds.data_vars))
    print("Dimensions:", dict(ds.dims))
    times = pd.to_datetime(ds['time'].values)
    print(f"Time: {times[0].date()} → {times[-1].date()} ({len(times)} steps)")

    # Auto-detect SWH variable
    for v in ['VHM0', 'hs', 'swh', 'SWH', 'Hs']:
        if v in ds.data_vars:
            arr = ds[v].values
            print(f"SWH variable '{v}': shape={arr.shape}, range=[{np.nanmin(arr):.3f}, {np.nanmax(arr):.3f}]m")
            break
    ds.close()

In [ ]:
# Cell 4 — Merge the two files

print("Opening both files...")
ds1 = xr.open_dataset(FILE_1)
ds2 = xr.open_dataset(FILE_2)

# ── Verify there is no time overlap and no gap ─────────────────
t1 = pd.to_datetime(ds1['time'].values)
t2 = pd.to_datetime(ds2['time'].values)

print(f"File 1 time: {t1[0].date()} → {t1[-1].date()} ({len(t1)} days)")
print(f"File 2 time: {t2[0].date()} → {t2[-1].date()} ({len(t2)} days)")

# Check for overlap
overlap = set(t1.date) & set(t2.date)
if overlap:
    print(f"⚠️ WARNING: {len(overlap)} overlapping dates detected.")
    print("  xr.concat will keep duplicates. They will be dropped below.")
else:
    print("✓ No overlap between the two files.")

# Check for gap
expected_next = t1[-1] + pd.Timedelta(days=1)
actual_next   = t2[0]
gap_days = (actual_next - expected_next).days
if gap_days == 0:
    print("✓ Files are continuous — no gap.")
elif gap_days > 0:
    print(f"⚠️ Gap of {gap_days} day(s) between files (will be NaN after merge — OK).")
elif gap_days < 0:
    print(f"⚠️ Files overlap by {-gap_days} day(s).")

# ── Merge along time dimension ──────────────────────────────────
print("\nMerging...")
ds_merged = xr.concat([ds1, ds2], dim='time')

# Drop duplicate timestamps if any overlap
_, unique_idx = np.unique(ds_merged.time.values, return_index=True)
if len(unique_idx) < len(ds_merged.time):
    n_dropped = len(ds_merged.time) - len(unique_idx)
    print(f"Dropping {n_dropped} duplicate timestamps...")
    ds_merged = ds_merged.isel(time=unique_idx)

# Sort by time (should already be sorted, but ensure it)
ds_merged = ds_merged.sortby('time')

t_merged = pd.to_datetime(ds_merged['time'].values)
print(f"\nMerged: {t_merged[0].date()} → {t_merged[-1].date()} ({len(t_merged)} days)")
print(f"Expected ~1096 days (3 years). Got: {len(t_merged)}")

# ── Add metadata ────────────────────────────────────────────────
ds_merged.attrs['merged_from'] = f"{os.path.basename(FILE_1)} + {os.path.basename(FILE_2)}"
ds_merged.attrs['merge_date']  = pd.Timestamp.now().strftime('%Y-%m-%d')
ds_merged.attrs['description'] = 'Daily SWH — Indian Ocean region 2021-2023 (merged)'

# ── Save ────────────────────────────────────────────────────────
print(f"\nSaving to: {OUT_FILE}")
ds_merged.to_netcdf(OUT_FILE, encoding={
    v: {'zlib': True, 'complevel': 4}
    for v in ds_merged.data_vars
})
size_mb = os.path.getsize(OUT_FILE)/1024**2
print(f"✅ Saved ({size_mb:.1f} MB)")
ds1.close(); ds2.close()

In [ ]:
# Cell 5 — Verify merged file + check all 5 locations
print("Verifying merged file...")
ds = xr.open_dataset(OUT_FILE)
times_index = pd.to_datetime(ds['time'].values)

# Auto-detect variable
SWH_VAR = None
for v in ['VHM0', 'hs', 'swh', 'SWH', 'Hs']:
    if v in ds.data_vars: SWH_VAR = v; break
print(f"SWH variable: {SWH_VAR}")
print(f"Period: {times_index[0].date()} → {times_index[-1].date()} ({len(times_index)} days)")

LOCATIONS = {
    'Arabian_Sea':   (15.0, 65.0),
    'Bay_of_Bengal': (12.0, 87.0),
    'Andaman_Sea':   (11.0, 95.0),
    'Lakshadweep':   (10.0, 73.0),
    'South_IO':      (-5.0, 75.0),
}

print("\nLocation check:")
all_ok = True
for loc, (lat, lon) in LOCATIONS.items():
    swh = ds[SWH_VAR].sel(latitude=lat, longitude=lon, method='nearest').values.flatten()
    nan_pct = np.isnan(swh).mean()*100
    valid = swh[~np.isnan(swh)]
    status = "✓" if nan_pct < 20 else "⚠️ HIGH NaN — consider shifting coordinates"
    if nan_pct >= 20: all_ok = False
    print(f"  {loc:<18} | NaN={nan_pct:.1f}% | mean={np.nanmean(swh):.2f}m | "
          f"max={np.nanmax(swh):.2f}m | {status}")

print()
if all_ok:
    print("✅ All locations have < 20% NaN. Ready for model training.")
    print(f"   Use DATA_FILE = '{OUT_FILE}' in all model notebooks.")
else:
    print("⚠️  Some locations have high NaN. See note below.")
    print("   Options: (1) interpolate in Cell 2 of each notebook (already handled)")
    print("            (2) shift coordinates 1-2 degrees offshore for high-NaN locations")

# ── Visualise full merged series ──────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for ax, (loc, (lat, lon)) in zip(axes, LOCATIONS.items()):
    swh = ds[SWH_VAR].sel(latitude=lat, longitude=lon, method='nearest').values.flatten()
    swh_s = pd.Series(swh, index=times_index)
    ax.plot(times_index, swh, lw=0.5, color='steelblue', alpha=0.7)
    ax.plot(times_index, swh_s.rolling(30, center=True).mean(), 'r-', lw=1.2, label='30d MA')
    # Mark the original file boundary
    boundary = t1[-1] if 't1' in dir() else times_index[int(len(times_index)//2)]
    ax.axvline(boundary, color='orange', lw=1.5, ls='--', label='File boundary')
    ax.set_title(f"{loc} | μ={np.nanmean(swh):.2f}m | max={np.nanmax(swh):.2f}m", fontsize=8)
    ax.set_ylabel('SWH (m)', fontsize=7); ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc='upper right')
fig.suptitle('Merged SWH — 5 Indian Ocean Locations 2021-2023\n'
             'Orange = original file boundary. No discontinuity = good merge.', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'plot_swh_merged_verification.png'), dpi=120, bbox_inches='tight')
plt.show()
print("\nIf the series is continuous at the orange line, the merge is correct.")
ds.close()